在本节中，将使用iris（鸢尾花）数据集实现一个简单的分类器。
如前一节所介绍的，iris数据集一共有4个特征值：分别是：sepal length（萼片长度）、sepal width（萼片宽度）、petal length（花瓣长度）、petal width （花瓣宽度）。根据这些数据鸢尾花被分为了3个类：Setosa、Versicolour 和 Virginica，样本数量150个，每类50个。在下面的示例中，为了可视化绘图方便，我们只取了4个特征中的2个（petal length、petal width）用于计算。

In [7]:
import numpy as np
import pandas as pd
iris = pd.read_csv(r"..\DataSet\iris_data.csv")

X = iris.filter(["sepal_length","sepal_width"])
y = iris.filter(["species"])
print("Class labels: ",np.unique(y))

Class labels:  ['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']


In [8]:
y = y['species'].astype('category')
y = y.cat.codes
y.head()

0    0
1    0
2    0
3    0
4    0
dtype: int8

和上一节同样，把数据集分成训练集和测试集：

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)

我们随机的将数据分为2份，其中30%（45条记录）用作测试数据，70%（105条记录）用作训练数据。train_test_split 函数在拆分之前已经随机的打乱了数据。否则，来自'Iris-setosa'类和'Iris-versicolor'类的所有记录都将出现在训练数集中；而测试数据集将包含来自'Iris-virginica'类的45个示例。通过random_state参数，使用内部伪随机数生成器，在拆分之前对数据集进行乱序。而使用一个固定random_state值可以确保我们的结果是可重现的，也就是说random_state实际上是内部伪随机数生成器的种子。最后，通过 layerify = y 利用了对分层的内置支持。在这种情况下，分层意味着train_test_split方法返回与输入数据集具有相同比例的类标签的训练和测试子集。可以使用NumPy的bincount函数计算数组中每个值的出现次数，以验证是否确实如此：

In [11]:
print(np.bincount(y))
print(np.bincount(y_train))
print(np.bincount(y_test))

[50 50 50]
[35 35 35]
[15 15 15]


和上一节一样，在训练之前还需要对数据进行归一化，统一缩放：

In [12]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

接下来我们使用Skelearn中的感知器进行模型的训练。在训练之前，先了解2个名词：（1）学习率（learn rate）简单说就是学习的速率。一个模型需要对此的训练，通过每次训练的反馈结果来调整参数。粗略的讲，学习率决定了调整的“步伐”大小。学习率是一个超参数，在训练开始值需要指定。学习率过小，会使得收敛的过程变慢，效率不高，同时也有可能陷于局部的极值或鞍点；学习率过大，则有可能越过全局的最优点。（2）Epoch，一个epoch就是使用训练集中的全部样本训练一次。通俗的讲，Epoch的值就是整个训练数据集被反复使用几次。Epoch数是一个超参数，它定义了学习算法在整个训练数据集中的工作次数。一个Epoch意味着训练数据集中的每个样本都有机会更新内部模型参数。同时也可以直观的看到，学习率越小，需要的epoch就越多。
一般而言，简单的感知器是一个二分类的。但是在skeleran中已经做了处理，使其可以进行多分类。下面对感知器进行训练：

In [13]:
from sklearn.linear_model import Perceptron
ppn = Perceptron(eta0=0.1, random_state=1)
ppn.fit(X_train, y_train)

Perceptron(eta0=0.1, random_state=1)

从sklearn.linear_model模块加载Perceptron类后，初始一个新的Perceptron对象并通过fit方法训练模型。在这里，模型参数 eta0 相当于学习率，而n_iter参数定义了epoch的数量（通过训练数据集）。找到合适的学习率需要一些实验。如果学习率太大，算法会超过全局代价最小值。如果学习率太小，算法将需要更多的epoch才能收敛，这会使学习速度变慢——尤其是对于大型数据集。此外，我们使用 random_state参数来确保每个epoch之后训练数据集的初始混洗的可重复性。
在 scikit-learn 中训练了一个模型后，我们可以通过 predict 方法进行预测：

In [14]:
y_pred = ppn.predict(X_test)
#print(y_test.values.ravel())
print('Misclassified examples: %d' % (y_test!= y_pred).sum())


Misclassified examples: 14


In [15]:
from sklearn.metrics import accuracy_score
print('Accuracy: %.3f' % accuracy_score(y_test, y_pred))


Accuracy: 0.689
